# 06 — Encoding Categorical Features

**Difficulty:** ⭐⭐⭐ &nbsp;&nbsp;|&nbsp;&nbsp; **Estimated Time:** 3 hours  
**Theme:** Credit Card Fraud Detection  
**Libraries:** numpy, pandas, matplotlib, seaborn, scikit-learn, category_encoders

---

## Learning Objectives
By the end of this notebook you will be able to:
1. Explain why categorical variables must be encoded before ML algorithms can use them
2. Apply **Ordinal Encoding** correctly — and know when NOT to use it
3. Apply **One-Hot Encoding** and avoid the dummy variable trap
4. Implement **Binary Encoding** for high-cardinality columns
5. Implement **Target Encoding** and protect against data leakage with cross-fitting
6. Handle **unseen categories** at inference time

---
## 1. Why Does This Matter for ML?

ML models are fundamentally **mathematical functions**. They multiply weights by feature values, compute distances, and optimise numeric loss functions. They cannot process the string `"FOOD"` or `"TRAVEL"` — those have no mathematical meaning.

In a credit card fraud detection system, you might have:
- `merchant_category`: FOOD, TRAVEL, ELECTRONICS, CLOTHING, ... (15–20 categories)
- `device_type`: mobile, desktop, tablet (3 categories)
- `risk_level`: low, medium, high (3 ordered categories)
- `country`: 30–50 countries

**Every single one of these must be converted into numbers before any ML algorithm can use them.**

The encoding method you choose matters enormously — the wrong choice can:
- Imply false ordering (banana > apple is meaningless)
- Explode your feature space (50 countries → 49 columns)
- Leak information from the test set (target encoding done naively)

---
## 2. Real-World Analogy: Translating a Foreign Menu

Imagine you are handed a restaurant menu written entirely in Japanese. You need to calculate the average price of dishes. Your calculator cannot process Japanese characters — you first need to translate each dish name into a price number.

But **how** you translate matters:
- **Ordinal translation:** Some dishes come in sizes (small=1, medium=2, large=3). The order matters.
- **One-hot translation:** Dishes belong to categories (sushi, ramen, tempura) with no ranking. You create a separate "is this dish sushi?" column for each category.
- **Target encoding:** Replace each cuisine type with its average customer satisfaction score.

Using the **wrong** translation method is like pricing a dish based on alphabetical order of its name — the numbers are technically there, but they are meaningless garbage.

This is exactly what happens when you apply ordinal encoding to nominal categories (FOOD=1, TRAVEL=2, ELECTRONICS=3 implies ELECTRONICS is 3× "more" than FOOD — which is nonsense).

---
## 3. Setup — Imports & Synthetic Dataset

In [ ]:
# ── Core libraries ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')  # suppress category_encoders deprecation warnings

# ── Scikit-learn tools ────────────────────────────────────────────────────────
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# ── category_encoders library (install if needed: pip install category-encoders)
try:
    import category_encoders as ce
    CE_AVAILABLE = True
    print('category_encoders is available.')
except ImportError:
    CE_AVAILABLE = False
    print('category_encoders not installed. Run: pip install category-encoders')
    print('Binary encoding will use manual numpy implementation instead.')

# ── Reproducibility & aesthetics ─────────────────────────────────────────────
np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print('Setup complete.')

In [ ]:
# ── Generate synthetic credit-card transaction dataset ────────────────────────
# 3000 transactions with categorical features relevant to fraud detection

n = 3000  # total transactions

# ── Define category lists ─────────────────────────────────────────────────────
merchant_categories = [
    'FOOD', 'TRAVEL', 'ELECTRONICS', 'CLOTHING', 'GROCERIES',
    'GAS_STATION', 'PHARMACY', 'ENTERTAINMENT', 'UTILITIES', 'HEALTHCARE',
    'GAMBLING', 'CRYPTO', 'JEWELRY', 'SPORTS', 'HOME_IMPROVEMENT'
]  # 15 categories

device_types = ['mobile', 'desktop', 'tablet']  # 3 nominal categories
risk_levels  = ['low', 'medium', 'high']          # 3 ORDINAL categories (order matters!)

# 30 country codes for binary encoding demonstration
countries = [
    'US', 'UK', 'DE', 'FR', 'CA', 'AU', 'JP', 'CN', 'IN', 'BR',
    'MX', 'RU', 'ZA', 'NG', 'EG', 'KR', 'IT', 'ES', 'NL', 'SE',
    'NO', 'DK', 'PL', 'CZ', 'AR', 'CL', 'CO', 'PE', 'PH', 'ID'
]  # 30 countries

# ── Fraud probability varies by merchant category (reflects real patterns) ────
# High-risk categories have higher fraud rates
fraud_rates = {
    'FOOD': 0.03, 'TRAVEL': 0.08, 'ELECTRONICS': 0.12, 'CLOTHING': 0.05,
    'GROCERIES': 0.02, 'GAS_STATION': 0.07, 'PHARMACY': 0.04, 'ENTERTAINMENT': 0.09,
    'UTILITIES': 0.03, 'HEALTHCARE': 0.04, 'GAMBLING': 0.18, 'CRYPTO': 0.22,
    'JEWELRY': 0.15, 'SPORTS': 0.06, 'HOME_IMPROVEMENT': 0.05
}

# ── Randomly assign categorical features ─────────────────────────────────────
# merchant_category: roughly uniform across categories
merchant_col = np.random.choice(merchant_categories, size=n)

# device_type: mobile is most common (60%), desktop (30%), tablet (10%)
device_col   = np.random.choice(device_types, size=n, p=[0.60, 0.30, 0.10])

# risk_level: mostly low (50%), some medium (35%), few high (15%)
risk_col     = np.random.choice(risk_levels, size=n, p=[0.50, 0.35, 0.15])

# country: US most common, others roughly uniform
country_probs = np.array([0.30] + [0.70/29]*29)  # US=30%, others share 70%
country_col   = np.random.choice(countries, size=n, p=country_probs)

# ── Generate the fraud label based on merchant category fraud rate ────────────
is_fraud = np.array(
    [1 if np.random.random() < fraud_rates[m] else 0 for m in merchant_col]
)

# Adjust: high-risk devices/risk-levels also increase fraud probability
device_fraud_boost = {'mobile': 0.03, 'desktop': 0.00, 'tablet': 0.01}
risk_fraud_boost   = {'low': 0.00,    'medium': 0.02,   'high': 0.07}
for i in range(n):
    if is_fraud[i] == 0:  # only add chance of flipping to fraud for non-fraud rows
        boost = device_fraud_boost[device_col[i]] + risk_fraud_boost[risk_col[i]]
        if np.random.random() < boost:
            is_fraud[i] = 1

# ── Assemble DataFrame ────────────────────────────────────────────────────────
df = pd.DataFrame({
    'merchant_category': merchant_col,
    'device_type'      : device_col,
    'risk_level'       : risk_col,
    'country'          : country_col,
    'is_fraud'         : is_fraud
})

# Shuffle rows
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f"Fraud rate: {df['is_fraud'].mean():.1%}")
print(f"Merchant categories: {df['merchant_category'].nunique()}")
print(f"Countries: {df['country'].nunique()}")
df.head(8)

---
## 4. Ordinal Encoding — For Features With a Natural Order

### Plain English
Some categories have a **natural ranking**: `low < medium < high`. Ordinal encoding assigns integers that **preserve this order**: `low=0, medium=1, high=2`.

### When to use
- The categories have a **meaningful order** (size, severity, rating, etc.)
- You want the model to learn that `high` is "more" than `medium`

### Critical danger — do NOT use for nominal categories
If you apply ordinal encoding to `merchant_category` (FOOD=1, TRAVEL=2, ELECTRONICS=3), you are telling the model ELECTRONICS is 3× FOOD and TRAVEL is halfway between them. This is **mathematically meaningless** and will mislead distance-based models.

**Ordinal = categories with a real ranked order  
Nominal = categories with no inherent rank**

In [ ]:
# ── Method 1: Manual ordinal encoding with pandas .map() ─────────────────────
# Explicitly define the mapping so the ordering is clear and controlled
risk_map = {'low': 0, 'medium': 1, 'high': 2}

df['risk_level_ordinal'] = df['risk_level'].map(risk_map)

print('=== Manual Ordinal Encoding (risk_level) ===')
print(df[['risk_level', 'risk_level_ordinal']].drop_duplicates()
        .sort_values('risk_level_ordinal'))

# ── Method 2: sklearn OrdinalEncoder ─────────────────────────────────────────
# categories parameter explicitly defines the order for each column
# This prevents sklearn from guessing alphabetical order (which would give
# high=0, low=1, medium=2 — completely wrong!)
ordinal_enc = OrdinalEncoder(
    categories=[['low', 'medium', 'high']],  # must be list-of-lists (one per column)
    handle_unknown='use_encoded_value',        # maps unseen values to NaN instead of crashing
    unknown_value=np.nan
)

# sklearn requires 2-D input
risk_2d = df[['risk_level']]
risk_encoded_sklearn = ordinal_enc.fit_transform(risk_2d)

print('\n=== sklearn OrdinalEncoder (risk_level) ===')
comparison = pd.DataFrame({
    'risk_level'       : df['risk_level'],
    'manual_encoding'  : df['risk_level_ordinal'],
    'sklearn_encoding' : risk_encoded_sklearn.ravel().astype(int)
}).drop_duplicates().sort_values('manual_encoding')
print(comparison)
print('\nBoth methods produce the same result: low=0, medium=1, high=2')
print('The ordering preserves the semantic meaning of the risk levels.')

In [ ]:
# ── Demonstrate the danger of ordinal encoding on nominal data ────────────────
# What happens if you mistakenly apply ordinal encoding to merchant_category?

# Default OrdinalEncoder with no explicit category order = alphabetical order
bad_ordinal = OrdinalEncoder()  # no categories specified!
merchant_bad = bad_ordinal.fit_transform(df[['merchant_category']])

# Show the arbitrary alphabet-based encoding
bad_mapping = pd.DataFrame({
    'merchant_category' : bad_ordinal.categories_[0],
    'ordinal_code'      : range(len(bad_ordinal.categories_[0]))
})
print('=== BAD: Ordinal encoding on a nominal feature (merchant_category) ===')
print(bad_mapping.to_string(index=False))
print()
print('PROBLEM: This encoding implies:')
print('  CRYPTO (2) - CLOTHING (1) = 1')
print('  TRAVEL (13) is 6.5× FOOD (4)')
print('  UTILITIES (14) > JEWELRY (7) — which is meaningless!')
print()
print('For nominal categories, use One-Hot Encoding instead.')

---
## 5. One-Hot Encoding (OHE) — For Nominal Categories

### Plain English
Create a **separate binary column for each category**. For each row, exactly one column is 1 (the category the row belongs to) and all others are 0.

### Example
| device_type | mobile | tablet |
|---|---|---|
| mobile | 1 | 0 |
| desktop | 0 | 0 |
| tablet | 0 | 1 |

Notice: with 3 categories, we only need **2 columns** (we drop `desktop`). If mobile=0 and tablet=0, the device must be desktop. This is the **dummy variable trap** fix.

### The Dummy Variable Trap
If you create n columns for n categories, one column is perfectly predictable from the others. This is **perfect multicollinearity**, which breaks linear models (the matrix becomes non-invertible). Always use `n-1` columns: `drop='first'` in sklearn.

### Cardinality explosion problem
50 countries → 49 new columns  
1,000 zip codes → 999 new columns  
High-cardinality features make OHE impractical.

In [ ]:
# ── One-Hot Encoding with sklearn ─────────────────────────────────────────────
# Step 1: Show the dummy variable trap (n columns for n categories)

ohe_with_trap = OneHotEncoder(
    drop=None,           # NO dropping — creates n columns (bad for linear models)
    sparse_output=False  # return dense array instead of sparse matrix
)
device_3cols = ohe_with_trap.fit_transform(df[['device_type']])

print('=== OHE WITH dummy variable trap (drop=None) ===')
print(f'Columns created: {ohe_with_trap.get_feature_names_out()}')
print(f'Shape: {device_3cols.shape}  ← 3 columns for 3 categories')
# Prove the multicollinearity: the three columns always sum to 1.0
print(f'\nRow sums (should all be 1.0): {device_3cols.sum(axis=1)[:5]}')
print('The third column = 1 - col1 - col2, so it carries NO new information.')

print()
# Step 2: Correct OHE dropping the first category
ohe_correct = OneHotEncoder(
    drop='first',        # drop the first category (alphabetically) — avoids multicollinearity
    sparse_output=False
)
device_2cols = ohe_correct.fit_transform(df[['device_type']])

print('=== OHE WITHOUT dummy variable trap (drop=\'first\') ===')
print(f'Columns created: {ohe_correct.get_feature_names_out()}')
print(f'Shape: {device_2cols.shape}  ← 2 columns for 3 categories (correct!)')
print(f'Categories found: {ohe_correct.categories_}')
print(f"Dropped (reference) category: '{ohe_correct.categories_[0][0]}'")
print('When both columns are 0, the device_type is implicitly the dropped category.')

In [ ]:
# ── One-Hot Encoding with pandas get_dummies ──────────────────────────────────
# pandas version is often simpler for DataFrames; produces the same result

# Apply OHE to device_type using pandas — drop_first=True drops the reference category
device_dummies = pd.get_dummies(
    df['device_type'],
    prefix='device',   # column names will be: device_mobile, device_tablet, etc.
    drop_first=True    # equivalent to drop='first' in sklearn
)

print('=== pd.get_dummies() result (first 8 rows) ===')
combined = pd.concat([df['device_type'], device_dummies], axis=1)
print(combined.drop_duplicates(subset='device_type').sort_values('device_type').to_string())

print(f'\nNew columns: {list(device_dummies.columns)}')
print(f'Shape: {device_dummies.shape}')

# ── Show cardinality explosion for merchant_category ────────────────────────
merchant_dummies = pd.get_dummies(df['merchant_category'], prefix='merch', drop_first=True)
print(f'\n=== Cardinality: merchant_category ===')
print(f'Original: 1 column with {df["merchant_category"].nunique()} unique categories')
print(f'After OHE: {merchant_dummies.shape[1]} columns')
print(f'With 50 countries: would be 49 columns')
print(f'With 1000 zip codes: would be 999 columns')
print('For high-cardinality features, OHE is impractical — use Binary or Target Encoding.')

---
## 6. Binary Encoding — For High-Cardinality Nominal Features

### Plain English
Binary encoding is a two-step process:
1. **Assign each category an integer:** France=33, USA=1, Japan=5, ...
2. **Convert that integer to binary:** 33 → `100001` → columns [1, 0, 0, 0, 0, 1]

### Why it is more efficient than One-Hot

| Method | 30 countries needs... |
|---|---|
| One-Hot | 29 columns |
| Binary | ceil(log₂(30)) = **5 columns** |

Binary encoding represents n categories in only **log₂(n)** columns instead of n-1.

### When to use
- **High cardinality** (many categories): countries, zip codes, product IDs
- When you want fewer features than OHE but more separation than ordinal

### Limitation
The binary codes are not entirely interpretable (bit 3 of 5 doesn't have semantic meaning), but they work well with tree-based models.

In [ ]:
# ── Manual Binary Encoding Implementation ─────────────────────────────────────
# We implement this from scratch so you understand exactly what is happening.

def manual_binary_encode(series, col_name='feature'):
    """
    Binary-encode a pandas Series of categorical values.
    Step 1: map each unique category to an integer (0, 1, 2, ...)
    Step 2: convert each integer to its binary representation
    Step 3: return a DataFrame with one column per binary bit
    """
    # Step 1: assign integer codes to each unique category
    unique_cats = sorted(series.unique())   # sort for reproducibility
    cat_to_int  = {cat: i for i, cat in enumerate(unique_cats)}
    int_codes   = series.map(cat_to_int).values  # array of integers

    # Step 2: determine how many bits we need
    n_bits = max(1, int(np.ceil(np.log2(len(unique_cats) + 1))))
    # e.g., 30 categories → ceil(log2(31)) = 5 bits

    # Step 3: convert each integer to its binary representation
    # np.binary_repr(n, width=n_bits) returns e.g. '00101' for 5 with width=5
    bit_strings = [np.binary_repr(code, width=n_bits) for code in int_codes]

    # Step 4: split each bit string into individual columns
    bit_matrix = np.array([[int(b) for b in s] for s in bit_strings])

    # Build a DataFrame with meaningful column names
    col_names = [f'{col_name}_bit{n_bits - 1 - i}' for i in range(n_bits)]
    return pd.DataFrame(bit_matrix, columns=col_names, index=series.index), cat_to_int


# Apply to the `country` column
country_binary, country_mapping = manual_binary_encode(df['country'], col_name='country')

print(f'=== Manual Binary Encoding: country ===')
print(f'Original:  1 column, {df["country"].nunique()} unique values')
print(f'Binary:    {country_binary.shape[1]} columns  (log2({df["country"].nunique()}) ≈ {np.log2(df["country"].nunique()):.2f} → {country_binary.shape[1]} bits)')
print()
# Show a few examples
sample_countries = ['US', 'UK', 'DE', 'FR', 'JP', 'ZA']
print('Sample encodings:')
for c in sample_countries:
    idx = df[df['country'] == c].index[0]  # find one row with this country
    bits = country_binary.loc[idx].values
    int_code = country_mapping[c]
    print(f'  {c:>4} → int={int_code:>2} → binary={np.binary_repr(int_code, width=5)} → {bits}')

In [ ]:
# ── Use category_encoders BinaryEncoder if available ─────────────────────────
if CE_AVAILABLE:
    be = ce.BinaryEncoder(cols=['country'], return_df=True)
    country_ce = be.fit_transform(df[['country']])

    print('=== category_encoders BinaryEncoder: country ===')
    print(f'Input:  1 column')
    print(f'Output: {country_ce.shape[1]} columns')
    print(country_ce.head(5))
else:
    print('category_encoders not available — using manual implementation above.')

# ── Visual comparison: OHE vs Binary column counts ───────────────────────────
cardinalities    = [3, 5, 10, 15, 20, 30, 50, 100]
ohe_cols         = [c - 1 for c in cardinalities]          # n-1 columns for OHE
binary_cols      = [int(np.ceil(np.log2(c + 1))) for c in cardinalities]  # log2(n) for binary

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cardinalities, ohe_cols,    'o-', color='#e74c3c', linewidth=2, label='One-Hot (n−1 columns)')
ax.plot(cardinalities, binary_cols, 's-', color='#27ae60', linewidth=2, label='Binary (log₂n columns)')
ax.set_xlabel('Number of unique categories', fontsize=12)
ax.set_ylabel('Number of new columns created', fontsize=12)
ax.set_title('Column Count: One-Hot vs Binary Encoding', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(cardinalities)

# Annotate the 30-country case
ax.annotate('30 countries:\nOHE=29 cols\nBinary=5 cols',
            xy=(30, 29), xytext=(40, 35),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10, bbox=dict(boxstyle='round', fc='lightyellow'))
plt.tight_layout()
plt.show()

---
## 7. Target Encoding — Using the Label to Encode Categories

### Plain English
Replace each category with a **statistical summary of the target variable** for that category. For fraud detection:
- GAMBLING → average fraud rate for gambling transactions → 0.18
- GROCERIES → average fraud rate for grocery transactions → 0.02
- CRYPTO → average fraud rate for crypto transactions → 0.22

### Why it works
Target encoding directly captures how predictive each category is. It turns a categorical string into a number that carries real fraud-rate signal — ideal for tree-based models with high-cardinality features.

### THE DANGER: Massive Data Leakage
If you compute the fraud rate on the **same data you train on** (without any protection), your model learns that GAMBLING → 0.18 because it already saw those labels. When evaluated, the model appears brilliant but will fail on new data.

This is not just a theoretical concern — target-leaked features can produce near-perfect training accuracy and mediocre production accuracy.

### Solution: Cross-Fitting (Leave-One-Out)
Compute the mean using **all rows EXCEPT the current one** (leave-one-out), or use K-fold cross-validation to compute means on held-out folds. `category_encoders.TargetEncoder` does this by default.

In [ ]:
# ── Visualise: bar chart of fraud rate by merchant_category ──────────────────
# This IS target encoding — replace each category with the mean target value

fraud_by_merchant = (df.groupby('merchant_category')['is_fraud']
                       .agg(['mean', 'count'])
                       .rename(columns={'mean': 'fraud_rate', 'count': 'n_transactions'})
                       .sort_values('fraud_rate', ascending=True)
                       .reset_index())

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(fraud_by_merchant['merchant_category'],
               fraud_by_merchant['fraud_rate'] * 100,
               color=plt.cm.RdYlGn_r(fraud_by_merchant['fraud_rate'] / fraud_by_merchant['fraud_rate'].max()))

# Add the actual fraud rate label at the end of each bar
for bar, rate in zip(bars, fraud_by_merchant['fraud_rate']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f'{rate:.1%}', va='center', fontsize=9)

ax.set_xlabel('Fraud Rate (%)', fontsize=12)
ax.set_title('Fraud Rate by Merchant Category\n'
             '(Each bar shows the target-encoded value for that category)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, 28)
plt.tight_layout()
plt.show()

print('This bar chart visualises exactly what target encoding computes.')
print('CRYPTO and GAMBLING are replaced by their fraud rates (~0.22, ~0.18).')
print('These values become the encoded feature values in the ML input.')

In [ ]:
# ── Target Encoding: Naive (LEAKY) approach ────────────────────────────────
# WRONG: compute the mean fraud rate on ALL data, then split
# The test set's own fraud rates contaminate the encoding.

print('=== WRONG: Naive target encoding (data leakage) ===')

# Compute mean fraud rate per merchant_category on the ENTIRE dataset
global_mean = df['is_fraud'].mean()  # overall fraud rate (used for smoothing)
target_means_naive = df.groupby('merchant_category')['is_fraud'].mean()

# Replace each merchant_category with its fraud rate
df['merchant_target_LEAKY'] = df['merchant_category'].map(target_means_naive)

# NOW split — but the encoding already used test set labels!
X_cols = ['merchant_target_LEAKY', 'risk_level_ordinal']
X_leaky = df[X_cols].values
y_all   = df['is_fraud'].values

X_tr_lk, X_te_lk, y_tr_lk, y_te_lk = train_test_split(
    X_leaky, y_all, test_size=0.2, random_state=42, stratify=y_all
)
lr_leaky = LogisticRegression(max_iter=500, random_state=42)
lr_leaky.fit(X_tr_lk, y_tr_lk)

train_auc_leaky = roc_auc_score(y_tr_lk, lr_leaky.predict_proba(X_tr_lk)[:, 1])
test_auc_leaky  = roc_auc_score(y_te_lk, lr_leaky.predict_proba(X_te_lk)[:, 1])
print(f'  Train AUC (leaky): {train_auc_leaky:.4f}')
print(f'  Test  AUC (leaky): {test_auc_leaky:.4f}')
print(f'  Gap (optimism):   {train_auc_leaky - test_auc_leaky:+.4f}')

In [ ]:
# ── Target Encoding: Cross-Fit (Correct) Approach ─────────────────────────
# Split FIRST, then compute means on training folds only.
# For each row in the training set, compute the mean EXCLUDING that row's own category
# contribution — this prevents the row from "seeing" its own label.

print('=== RIGHT: Cross-fit target encoding (no leakage) ===')

# First do a proper train/test split on the ORIGINAL data
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=df['is_fraud']
)
df_train = df.loc[train_idx].copy()
df_test  = df.loc[test_idx].copy()

# ── K-Fold cross-fit encoding for the training set ────────────────────────
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Allocate column for the cross-fit encoded values
df_train['merchant_target_CLEAN'] = np.nan

train_arr = df_train['merchant_category'].values
y_train_arr = df_train['is_fraud'].values

for fold_num, (fit_idx, transform_idx) in enumerate(kf.split(df_train)):
    # Compute mean fraud rate using only the FIT fold rows
    fold_means = (pd.Series(y_train_arr[fit_idx])
                    .groupby(pd.Series(train_arr[fit_idx]))
                    .mean())
    # Use those means to encode the TRANSFORM fold rows
    cats_to_encode = pd.Series(train_arr[transform_idx])
    # For categories not seen in the fit fold, fall back to the global mean
    encoded_vals = cats_to_encode.map(fold_means).fillna(global_mean).values
    df_train.iloc[transform_idx, df_train.columns.get_loc('merchant_target_CLEAN')] = encoded_vals

# ── Encode the TEST set using means computed on ALL training data ─────────
# At inference time, we use the full training set statistics
full_train_means = df_train.groupby('merchant_category')['is_fraud'].mean()
df_test['merchant_target_CLEAN'] = (df_test['merchant_category']
                                     .map(full_train_means)
                                     .fillna(global_mean))

# ── Train and evaluate on clean encoding ─────────────────────────────────
X_cols_clean = ['merchant_target_CLEAN', 'risk_level_ordinal']
# Need to ensure risk_level_ordinal is in both splits
df_train['risk_level_ordinal'] = df_train['risk_level'].map({'low': 0, 'medium': 1, 'high': 2})
df_test['risk_level_ordinal']  = df_test['risk_level'].map({'low': 0, 'medium': 1, 'high': 2})

X_tr_cl  = df_train[X_cols_clean].values
y_tr_cl  = df_train['is_fraud'].values
X_te_cl  = df_test[X_cols_clean].values
y_te_cl  = df_test['is_fraud'].values

lr_clean = LogisticRegression(max_iter=500, random_state=42)
lr_clean.fit(X_tr_cl, y_tr_cl)

train_auc_clean = roc_auc_score(y_tr_cl, lr_clean.predict_proba(X_tr_cl)[:, 1])
test_auc_clean  = roc_auc_score(y_te_cl, lr_clean.predict_proba(X_te_cl)[:, 1])
print(f'  Train AUC (clean): {train_auc_clean:.4f}')
print(f'  Test  AUC (clean): {test_auc_clean:.4f}')
print(f'  Gap (optimism):   {train_auc_clean - test_auc_clean:+.4f}')

print()
print(f'Comparison:')
print(f'  Leaky test AUC:  {test_auc_leaky:.4f}  ← inflated by leakage')
print(f'  Clean test AUC:  {test_auc_clean:.4f}  ← honest estimate')

In [ ]:
# ── Target Encoding with category_encoders (does cross-fitting automatically) ─
if CE_AVAILABLE:
    print('=== category_encoders TargetEncoder (cross-fitting built in) ===')
    te = ce.TargetEncoder(cols=['merchant_category'], smoothing=1.0)
    # smoothing: shrinks rare-category means toward the global mean
    # This reduces overfitting for categories with few transactions

    # Always: fit on training data only
    X_train_te = df_train[['merchant_category']].copy()
    X_test_te  = df_test[['merchant_category']].copy()
    y_train_te = df_train['is_fraud'].values

    # fit_transform on train: internally uses cross-fitting to prevent leakage
    X_train_encoded = te.fit_transform(X_train_te, y_train_te)
    X_test_encoded  = te.transform(X_test_te)  # uses full training means for test

    print(f'First 5 encoded training values (merchant_category):')
    print(pd.concat([
        df_train['merchant_category'].reset_index(drop=True).head(),
        X_train_encoded.reset_index(drop=True).head()
    ], axis=1).to_string(index=False))

    # Show what the encoder learned (mapping per category)
    ce_mapping = te.ordinal_encoder.mapping
    print('\ncategory_encoders.TargetEncoder handles cross-fitting automatically.')
    print('Use it instead of the manual KFold loop above in production code.')
else:
    print('category_encoders not available — manual cross-fit demonstrated above.')

---
## 8. Handling Unseen Categories at Inference Time

In production, a new category can appear that was never in your training data:
- A new country starts appearing in transactions
- A new merchant category is created (e.g., "NFT" emerges)

Each encoder handles this differently, and you must know the behaviour of each.

In [ ]:
# ── Simulate an unseen category at inference time ─────────────────────────────
# Imagine: at prediction time, a transaction comes from 'XZ' — a country
# not present in the training data at all.

print('=== Handling Unseen Categories ===')
print()

# ── OHE with handle_unknown='ignore' (recommended for production) ─────────────
ohe_prod = OneHotEncoder(
    drop='first',
    sparse_output=False,
    handle_unknown='ignore'  # unseen categories → all zeros in that row
)
# Fit on known countries
known_countries = df[['country']].head(2500)  # "training" subset
ohe_prod.fit(known_countries)

# Try to encode a new country 'XZ'
new_transaction = pd.DataFrame({'country': ['XZ', 'US', 'FR']})
encoded_new = ohe_prod.transform(new_transaction)

print("Encoding ['XZ', 'US', 'FR'] after training on known countries:")
result_df = pd.DataFrame(encoded_new, columns=ohe_prod.get_feature_names_out())
result_df.insert(0, 'original', ['XZ', 'US', 'FR'])
# Only show first 8 columns for readability
print(result_df.iloc[:, :8].to_string(index=False))
print()
print("'XZ' (unseen) → all zeros. This means it is treated like the reference/dropped category.")
print("'US' and 'FR' → normal one-hot encoding.")

print()
# ── OHE with handle_unknown='error' (default in older sklearn) ────────────────
ohe_strict = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='error')
ohe_strict.fit(known_countries)
try:
    ohe_strict.transform(new_transaction)
except ValueError as e:
    print(f"handle_unknown='error' raises ValueError for unseen 'XZ':")
    print(f"  {str(e)[:120]}...")

print()
# ── Target encoding: fallback to global mean for unseen categories ────────────
unseen_country = 'XZ'
te_val = full_train_means.get(unseen_country, global_mean)
print(f"Target encoding unseen country '{unseen_country}':")
print(f"  Falls back to global mean fraud rate: {te_val:.4f}")
print("  This is a safe default — we have no category-specific information.")

---
## 9. Encoding Method Comparison Table

| Encoding Method | When to Use | New Columns Created | Handles Unseen Categories | Leakage Risk |
|---|---|---|---|---|
| **Ordinal** | Ordered categories (low/med/high) | 0 (replaces 1) | Map to `NaN` or special code | None |
| **One-Hot** | Nominal, low cardinality (< 15 cats) | n − 1 | All zeros (`handle_unknown='ignore'`) | None |
| **Binary** | Nominal, high cardinality | log₂(n) | Assign new integer → binary | None |
| **Target** | Any nominal, high cardinality | 0 (replaces 1) | Global mean fallback | **HIGH** if naive — use cross-fitting |

### Quick Decision Guide
```
Category has natural ORDER?  
  YES → Ordinal Encoding
  NO  →  How many unique values?
           < 15  → One-Hot Encoding (with drop_first)
           15–100 → Binary Encoding (or Target Encoding with cross-fit)
           > 100  → Target Encoding (with cross-fitting) or Hashing Encoder
```

In [ ]:
# ── Final: put it all together in a complete preprocessing pipeline ────────────
# Combining multiple encoding strategies in one pipeline for the fraud dataset

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ── Define which columns get which treatment ──────────────────────────────────
# Ordinal: risk_level
ordinal_cols = ['risk_level']
ordinal_enc_final = OrdinalEncoder(
    categories=[['low', 'medium', 'high']],
    handle_unknown='use_encoded_value',
    unknown_value=-1  # unseen risk levels get -1
)

# One-hot: device_type (3 categories — manageable)
ohe_cols   = ['device_type']
ohe_final  = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Binary (manual): country (30 categories — use binary to keep columns low)
# We'll use OHE here for sklearn Pipeline compatibility; switch to BinaryEncoder
# from category_encoders if available
country_cols   = ['country']
country_enc    = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# ── ColumnTransformer applies different transformers to different columns ──────
preprocessor = ColumnTransformer(
    transformers=[
        ('ordinal',  ordinal_enc_final, ordinal_cols),
        ('ohe',      ohe_final,         ohe_cols),
        ('country',  country_enc,       country_cols),
    ],
    remainder='drop'  # drop any columns not explicitly listed
)

# ── Full pipeline: preprocess → scale → classify ─────────────────────────────
full_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('scaler',     StandardScaler()),   # scale all encoded features to z-scores
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# ── Train and evaluate ────────────────────────────────────────────────────────
X_pipe = df[['risk_level', 'device_type', 'country']]
y_pipe = df['is_fraud']

X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(
    X_pipe, y_pipe, test_size=0.2, random_state=42, stratify=y_pipe
)

full_pipeline.fit(X_tr_p, y_tr_p)
test_acc = accuracy_score(y_te_p, full_pipeline.predict(X_te_p))
test_auc = roc_auc_score(y_te_p, full_pipeline.predict_proba(X_te_p)[:, 1])

print('=== Complete Encoding + Scaling + Classification Pipeline ===')
print(f'  Test Accuracy: {test_acc:.4f}')
print(f'  Test AUC-ROC:  {test_auc:.4f}')
print()
# Show the number of features produced by the preprocessor
n_features_out = full_pipeline[:-1].transform(X_tr_p).shape[1]
print(f'  Input features:  {X_pipe.shape[1]} columns (risk_level, device_type, country)')
print(f'  After encoding:  {n_features_out} columns')
print()
print('This pipeline handles all encoding and scaling correctly — leakage-free.')

---
## 10. Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** A feature `satisfaction_score` has values: `'very_low'`, `'low'`, `'medium'`, `'high'`, `'very_high'`. Which encoding should you use and why? Write the code.

<details>
<summary>▶ Reveal Answer</summary>

**Use Ordinal Encoding** because `satisfaction_score` has a clear natural order:  
`very_low < low < medium < high < very_high`

One-Hot would lose this ordering information. The model would be unable to learn that `high` and `very_high` are closer to each other than `high` and `very_low`.

```python
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd

# Method 1: pandas .map() — explicit and readable
order_map = {'very_low': 0, 'low': 1, 'medium': 2, 'high': 3, 'very_high': 4}
df['satisfaction_encoded'] = df['satisfaction_score'].map(order_map)

# Method 2: sklearn OrdinalEncoder — use in pipelines
enc = OrdinalEncoder(
    categories=[['very_low', 'low', 'medium', 'high', 'very_high']]
)
df['satisfaction_encoded'] = enc.fit_transform(df[['satisfaction_score']]).ravel()
```

**Key:** Always specify the `categories` order explicitly in sklearn. Without it, the encoder defaults to alphabetical order which gives `high=0, low=1, medium=2, very_high=3, very_low=4` — completely wrong.
</details>

---

**Question 2:** A `zip_code` feature has 8,000 unique values. Why is one-hot encoding a bad idea here? What would you use instead, and why?

<details>
<summary>▶ Reveal Answer</summary>

**Why OHE is bad for zip codes:**
1. **Column explosion:** 8,000 categories → 7,999 new columns. Your model now has 7,999 binary features just for zip code. This massively increases memory usage, training time, and the risk of overfitting.
2. **Sparsity:** Each row has exactly one 1 and 7,998 zeros — the feature matrix is 99.99% zeros. Many algorithms struggle with extreme sparsity.
3. **Rare categories:** Many zip codes will appear only once or twice in your training data, making OHE columns essentially useless noise.

**Better alternatives:**

| Method | Why Good | Columns |
|---|---|---|
| **Binary Encoding** | log₂(8000) ≈ 13 columns — compact representation | ~13 |
| **Target Encoding** | Replaces zip code with local fraud rate — captures signal directly | 1 |
| **Frequency Encoding** | Replace with how often the zip appears (proxy for size/activity) | 1 |
| **Hashing Encoder** | Hash to a fixed-size vector (e.g., 16 columns) — handles new zip codes | 16 |
| **Feature Engineering** | Extract state or region from the zip code first, then encode | Varies |

For fraud detection, **Target Encoding** (with cross-fitting) is often the best choice because different zip codes have genuinely different fraud rates.
</details>

---

**Question 3:** You apply target encoding on the full training set (`df_train`), then split into train and validation. Why is this data leakage, and how would you detect it?

<details>
<summary>▶ Reveal Answer</summary>

**Why it is leakage:**

When you compute `merchant_category = mean(is_fraud)` on the full training set, every row's own label contributes to its own encoded value. For example:
- If the 5 GAMBLING rows all have `is_fraud=1`, GAMBLING gets encoded as 1.0
- But one of those 5 rows will end up in your validation set
- The model sees `merchant_category_encoded = 1.0` for that row **because** it is fraud
- The model hasn't learned a generalizable pattern; it has memorised the validation label

**How to detect it:**
```python
# The smoking gun: validation performance >> test performance on truly new data
# OR: train AUC and val AUC are both suspiciously high and nearly identical
# (when target encoding leaks, the val set looks like the train set)

# Simple test: compare target-encoded values with actual fraud rates
# Leaky: each row's encoded value perfectly predicts its own label
# Clean: cross-fit values are slightly noisier, derived from other rows
```

**The fix:**
1. Split BEFORE any target encoding
2. Compute category means on the training fold only
3. Use K-Fold cross-fitting for training rows (prevents a row from encoding itself)
4. Use `category_encoders.TargetEncoder` which handles this automatically
</details>

---

**Question 4:** After one-hot encoding `country` (30 countries), your train set has 29 country columns. At prediction time, a transaction arrives from `'XZ'` — a country not in training. What does sklearn's OHE do by default, and what are the options?

<details>
<summary>▶ Reveal Answer</summary>

**Default behaviour (before sklearn 1.0):**  
`handle_unknown='error'` — raises a `ValueError`. Your model crashes in production when it sees an unknown category. This is a common production failure mode.

**Recommended for production: `handle_unknown='ignore'`**  
```python
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
```
With `ignore`, an unknown category like `'XZ'` gets all-zero encoding. The model treats it the same as the dropped (reference) category. This is a safe fallback — not perfect, but prevents crashes.

**Alternative options:**

| Strategy | Behaviour | When to Use |
|---|---|---|
| `handle_unknown='ignore'` | All zeros for unseen category | General-purpose safe default |
| `handle_unknown='infrequent_if_exist'` | Maps rare/unseen to a catch-all column | When training data has infrequent categories |
| Target Encoding fallback | Unseen category → global mean fraud rate | When you have a meaningful default |
| Add an explicit 'OTHER' category during training | Known bucket for novel categories | When you expect new categories in production |

**Best practice:** During training, add an `'OTHER'` row to your encoder vocabulary so unseen categories map to a trained `'OTHER'` one-hot column rather than all zeros.
</details>

---
## 11. Quick Reference Card

```python
# ── Ordinal Encoding ──────────────────────────────────────────────────────
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder(categories=[['low', 'medium', 'high']])
X_encoded = enc.fit_transform(X_train[['risk_level']])

# pandas shortcut:
df['risk_encoded'] = df['risk_level'].map({'low': 0, 'medium': 1, 'high': 2})

# ── One-Hot Encoding ──────────────────────────────────────────────────────
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
X_encoded = ohe.fit_transform(X_train[['device_type']])

# pandas shortcut:
dummies = pd.get_dummies(df['device_type'], prefix='dev', drop_first=True)

# ── Binary Encoding ───────────────────────────────────────────────────────
import category_encoders as ce
be = ce.BinaryEncoder(cols=['country'])
X_encoded = be.fit_transform(X_train[['country']])
X_test_enc = be.transform(X_test[['country']])  # uses train mapping

# ── Target Encoding (with cross-fitting, leakage-safe) ─────────────────────
import category_encoders as ce
te = ce.TargetEncoder(cols=['merchant_category'], smoothing=1.0)
X_train_enc = te.fit_transform(X_train[['merchant_category']], y_train)
X_test_enc  = te.transform(X_test[['merchant_category']])

# ── ColumnTransformer for mixed encoding in a Pipeline ──────────────────────
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
ct = ColumnTransformer([
    ('ordinal', OrdinalEncoder(categories=[['low','medium','high']]), ['risk_level']),
    ('ohe',     OneHotEncoder(drop='first'), ['device_type']),
])
pipe = Pipeline([('enc', ct), ('clf', LogisticRegression())])
pipe.fit(X_train, y_train)
```